<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Impacto en la Generalización</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">El valor real del data augmentation no está en transformar por transformar, sino en mejorar cómo el modelo generaliza. Este notebook conecta explícitamente augmentation con rendimiento fuera del entrenamiento.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Añade una lectura crítica sobre cuándo la variabilidad artificial ayuda de verdad al modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A relacionar augmentation con robustez y generalización, no solo con cantidad aparente de datos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Más augmentations no garantizan mejor generalización si no están alineadas con variaciones plausibles del problema real.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Generalizar mejor significa mantener buen comportamiento ante datos nuevos que no son copias del entrenamiento.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Tensores de entrada y etiquetas
"""Este bloque crea tensores que representan las entradas y las etiquetas del conjunto de entrenamiento.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "entradas" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en un tensor columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo del entrenamiento.
Estos tensores permiten observar cómo el modelo se ajusta a los datos de entrenamiento."""

import torch

entradas = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0],
                         [5.0, 6.0]])

etiquetas = torch.tensor([0.0, 1.0, 0.0]).unsqueeze(1)

entradas, etiquetas # Muestra los tensores de entrada y las etiquetas con la dimensión adicional.


In [ ]:
# Bloque 2 — Tensores de validación
"""Este bloque crea tensores que representan las entradas y etiquetas del conjunto de validación.
La variable "entradas_val" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en un tensor columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas_val" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo de la validación.
Estos tensores permiten observar el rendimiento del modelo sobre muestras de validación no utilizadas para entrenarlo."""

entradas_val = torch.tensor([[2.0, 1.0],
                             [4.0, 3.0],
                             [6.0, 5.0]])

etiquetas_val = torch.tensor([1.0, 0.0, 1.0]).unsqueeze(1)

entradas_val, etiquetas_val # Muestra los tensores de validación con la dimensión adicional.


In [ ]:
# Bloque 3 — Definición de la red neuronal
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedGeneralizacion()" que representa un modelo de red neuronal simple.
La clase "RedGeneralizacion()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedGeneralizacion()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase  "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase  "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos la llamada a la función "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos la llamada a la función "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo tras aplicar las capas y la activación a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.
La variable "red" almacena una instancia de la clase "RedGeneralizacion()", representando el modelo de red neuronal definido.
La estructura permite analizar cómo el modelo se ajusta a los datos de entrenamiento y responde a datos de validación."""

import torch.nn as nn

class RedGeneralizacion(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedGeneralizacion()

red # Muestra la estructura del modelo.


In [ ]:
# Bloque 4 — Función de pérdida y optimizador
"""Este bloque define la función de pérdida y el optimizador utilizados durante el entrenamiento.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto calculador de la función de pérdida, que recibe los logits de la red y las etiquetas.

Aplicamos la clase "torch.optim.Adam()" a la variable "optimizador" para crear un objeto optimizador de parámetros de la red.
Pasamos como argumentos los parámetros de la red mediante la expresión "red.parameters()".
El método ".parameters()" devuelve un iterador sobre los parámetros de la red que serán actualizados por el optimizador.
El parámetro "lr=" especifica la tasa de aprendizaje utilizada por el optimizador; en este caso, 0.01.
El optimizador Adam actualiza los parámetros de la red utilizando los gradientes calculados."""

criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.Adam(red.parameters(), lr=0.01)

criterio, optimizador # Muestra la función de pérdida y el optimizador definidos.


In [ ]:
# Bloque 5 — Loop de entrenamiento con evaluación de generalización
"""Este bloque implementa el loop de entrenamiento y la evaluación de generalización.
La variable "epocas" define cuántas veces se repite el ciclo completo de entrenamiento; en este caso, 20 veces.
La variable "historial_train" se inicializa como una lista vacía "[]" para almacenar los valores de pérdida obtenidos durante el entrenamiento.
La variable "historial_val" se inicializa como una lista vacía "[]" para almacenar los valores de pérdida obtenidos durante la validación en cada época.

Primero, realizamos el loop de entrenamiento para actualizar los parámetros de la red.
Utilizamos un bucle "for" para iterar sobre el número de épocas definido en la variable "epocas"; es decir, en este caso se repetirá 20 veces.
La expresión "for _ in range(epocas):" inicia el bucle que se repetirá el número de veces definido en "epocas".
En cada iteración se ejecuta el forward pass, se calcula la pérdida, se reinician los gradientes previos, se ejecuta backward para obtener nuevos gradientes
y se actualizan los parámetros mediante el optimizador.

A continuación, realizamos la validación en la misma época para evaluar el rendimiento del modelo en el conjunto de datos de validación.
Utilizamos la palabra clave "with" para crear un contexto en el que el cálculo de gradientes está desactivado.
Aplicamos la clase "torch.no_grad()" para desactivar el cálculo de gradientes durante la validación en este contexto.
El contexto "with torch.no_grad():" desactiva el cálculo de gradientes para evitar el almacenamiento innecesario de información para backpropagation.
Este contexto no cambia el modelo a modo evaluación; esta red no contiene capas cuyo comportamiento dependa de ese modo.

En cada época se calcula la pérdida de entrenamiento antes de actualizar los parámetros y la pérdida de validación después de actualizarlos.
Además, los valores de pérdida de entrenamiento se almacenan en la lista "historial_train" y los valores de pérdida de validación 
se almacenan en la lista "historial_val" para su posterior análisis.
La comparación de ambas pérdidas orienta el análisis de generalización, aunque este ejemplo pequeño no aplica data augmentation ni compara su impacto."""

epocas = 20
historial_train = []
historial_val = []

for _ in range(epocas):
    pred_train = red(entradas)
    perdida_train = criterio(pred_train, etiquetas)
    optimizador.zero_grad()
    perdida_train.backward()
    optimizador.step()

    with torch.no_grad():
        pred_val = red(entradas_val)
        perdida_val = criterio(pred_val, etiquetas_val)

    historial_train.append(float(perdida_train.item()))
    historial_val.append(float(perdida_val.item()))



In [ ]:
# Bloque 6 — Resultados finales
"""Este bloque organiza los resultados del proceso de evaluación de generalización.
La variable "resultado" almacena en forma de diccionario las pérdidas de entrenamiento y validación por época.
La estructura permite observar la evolución de ambas pérdidas, sin demostrar por sí sola el impacto del data augmentation."""

resultado = {
    "perdida_entrenamiento": historial_train,
    "perdida_validacion": historial_val
}

resultado # Muestra el diccionario con las pérdidas de entrenamiento y validación por época.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que el data augmentation debe evaluarse por su efecto sobre la generalización y no solo por la variedad visual que introduce.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">La utilidad de una augmentación se mide por cómo ayuda al modelo fuera del conjunto de entrenamiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Asumir que cualquier incremento de diversidad visual es automáticamente beneficioso.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Implementación con Transforms para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Implementación con Transforms y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">08_Data_augmentation/04_La_implementacion_con_transforms.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">